# MONAI Lung CT Registration - Train and Register Model

Train a MONAI LocalNet model on Snowflake's GPU compute pool and log to the Model Registry.

## Model
- **Architecture**: LocalNet (deformable registration network)
- **Input**: Paired CT scans - moving (inspiration) + fixed (expiration)
- **Output**: Deformation displacement field (DDF)

## Prerequisites
- Run `01_setup.sql` and `02_ingest_data.ipynb` first

In [1]:
!pip install -q monai nibabel torch snowflake-ml-python

In [2]:
import os
import tempfile
import time
import gzip
import shutil

import torch
import torch.nn.functional as F
import numpy as np
import nibabel as nib

from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from snowflake.ml.jobs import remote

print(f"PyTorch: {torch.__version__}")

/opt/anaconda3/envs/snowml-local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.5.1


## Configuration

In [3]:
# Connection name from ~/.snowflake/connections.toml
# See: https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-session
CONNECTION_NAME = "monai-quickstart"  # Change to your connection name

# Snowflake objects created by 01_setup.sql
ROLE = "MONAI_USER"
WAREHOUSE = "MONAI_WH"
DATABASE = "MONAI_QUICKSTART_DB"
SCHEMA = "ML"

COMPUTE_POOL = "MONAI_GPU_POOL"
STAGE = "@MEDICAL_IMAGES_STG"
RESULTS_STAGE = "@MODEL_ARTIFACTS_STG"

# Training configuration
CONFIG = {
    "spatial_size": (96, 96, 96),
    "num_channel_initial": 32,
    "batch_size": 2,
    "max_epochs": 15,
    "learning_rate": 1e-4,
    "reg_weight": 1.0,
}

CT_MIN_HU = -1000
CT_MAX_HU = 500

In [4]:
# Create session using connection from connections.toml
session = Session.builder.config("connection_name", CONNECTION_NAME).create()

# Switch to MONAI resources
session.use_role(ROLE)
session.use_warehouse(WAREHOUSE)
session.use_database(DATABASE)
session.use_schema(SCHEMA)

# Create results stage if not exists
session.sql("CREATE STAGE IF NOT EXISTS MODEL_ARTIFACTS_STG").collect()

print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

Connected: "MONAI_QUICKSTART_DB"."ML"


## Prepare Training Data Paths

In [6]:
# List available expiration scans from stage
files_df = session.sql(f"LIST {STAGE}/scans/").to_pandas()
file_names = files_df.iloc[:, 0].tolist()  # First column contains file paths
exp_files = [f for f in file_names if "_exp.nii.gz" in f]
num_cases = len(exp_files)
print(f"Available cases: {num_cases}")

# Build paired file paths (exp=fixed, insp=moving)
all_files = []
for i in range(1, num_cases + 1):
    case_id = f"case_{i:03d}"
    all_files.append({
        "case_id": case_id,
        "fixed_path": f"{STAGE}/scans/{case_id}_exp.nii.gz",
        "moving_path": f"{STAGE}/scans/{case_id}_insp.nii.gz",
        "fixed_label_path": f"{STAGE}/lungMasks/{case_id}_exp.nii.gz",
        "moving_label_path": f"{STAGE}/lungMasks/{case_id}_insp.nii.gz",
    })

# Split train/val (80/20)
split_idx = max(1, int(0.8 * len(all_files)))
train_split = all_files[:split_idx]
val_split = all_files[split_idx:]

print(f"Train: {len(train_split)}, Val: {len(val_split)}")
print(f"Validation case: {val_split[0]['case_id']}")

Available cases: 20
Train: 16, Val: 4
Validation case: case_017


## Download Validation Data Locally

Download one validation case for sample input during model registration.

In [7]:
def download_and_decompress(stage_path, local_dir):
    """Download file from stage and handle double-gzip if needed."""
    os.makedirs(local_dir, exist_ok=True)
    session.file.get(stage_path, local_dir)
    
    # Handle double-gzip
    for f in os.listdir(local_dir):
        if f.endswith('.gz.gz'):
            gz_path = os.path.join(local_dir, f)
            out_path = gz_path[:-3]
            with gzip.open(gz_path, 'rb') as f_in:
                with open(out_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            os.remove(gz_path)


def preprocess_ct(nifti_path, spatial_size):
    """Load and preprocess CT scan matching training pipeline."""
    img = nib.load(nifti_path)
    data = img.get_fdata().astype(np.float32)
    
    # HU normalization (same as training: ScaleIntensityRanged)
    data = np.clip(data, CT_MIN_HU, CT_MAX_HU)
    data = (data - CT_MIN_HU) / (CT_MAX_HU - CT_MIN_HU)
    
    # Resize (same as training: Resized with trilinear mode)
    tensor = torch.from_numpy(data).unsqueeze(0).unsqueeze(0)  # [1, 1, D, H, W]
    tensor = F.interpolate(tensor, size=spatial_size, mode='trilinear', align_corners=True)
    
    return tensor.squeeze(0)  # [1, D, H, W]


# Download validation case for sample input
LOCAL_DATA_DIR = tempfile.mkdtemp()
val_case = val_split[0]

download_and_decompress(val_case["fixed_path"], os.path.join(LOCAL_DATA_DIR, "fixed"))
download_and_decompress(val_case["moving_path"], os.path.join(LOCAL_DATA_DIR, "moving"))

# Find downloaded files
fixed_file = [f for f in os.listdir(f"{LOCAL_DATA_DIR}/fixed") if f.endswith('.nii.gz')][0]
moving_file = [f for f in os.listdir(f"{LOCAL_DATA_DIR}/moving") if f.endswith('.nii.gz')][0]

LOCAL_FIXED_PATH = f"{LOCAL_DATA_DIR}/fixed/{fixed_file}"
LOCAL_MOVING_PATH = f"{LOCAL_DATA_DIR}/moving/{moving_file}"

print(f"Downloaded validation case: {val_case['case_id']}")
print(f"  Fixed: {LOCAL_FIXED_PATH}")
print(f"  Moving: {LOCAL_MOVING_PATH}")

Downloaded validation case: case_017
  Fixed: /var/folders/wh/1t1qbkgx1csgf4q8fxkt5gnr0000gn/T/tmpe8305tu4/fixed/case_017_exp.nii.gz
  Moving: /var/folders/wh/1t1qbkgx1csgf4q8fxkt5gnr0000gn/T/tmpe8305tu4/moving/case_017_insp.nii.gz


## Define Remote Training Function

This function runs on Snowflake's GPU compute pool.

In [11]:
@remote(
    compute_pool=COMPUTE_POOL,
    stage_name=RESULTS_STAGE,
    pip_requirements=["monai", "nibabel", "torch"],
    external_access_integrations=["PYPI_EAI"],
)
def train_model(train_files, val_files, config, save_stage):
    import os
    import tempfile
    import numpy as np
    import torch
    import torch.optim as optim
    
    from snowflake.snowpark.context import get_active_session
    from snowflake.snowpark.files import SnowflakeFile
    
    from monai.networks.nets import LocalNet
    from monai.networks.blocks import Warp
    from monai.losses import GlobalMutualInformationLoss, BendingEnergyLoss
    from monai.transforms import (
        Compose, LoadImaged, EnsureChannelFirstd,
        ScaleIntensityRanged, Resized
    )
    from monai.data import Dataset, DataLoader
    
    CT_MIN_HU, CT_MAX_HU = -1000, 500
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    def read_from_stage(path):
        clean = path if path.startswith("@") else f"@{path}"
        with SnowflakeFile.open(clean, 'rb') as f:
            return f.read()
    
    class StageDataset(Dataset):
        def __init__(self, files, transform):
            self.files = files
            self.transform = transform
        
        def __len__(self):
            return len(self.files)
        
        def __getitem__(self, idx):
            item = self.files[idx]
            temp_files = {}
            
            for key in ["fixed", "moving", "fixed_label", "moving_label"]:
                data = read_from_stage(item[f"{key}_path"])
                tf = tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False)
                tf.write(data)
                tf.close()
                temp_files[key] = tf.name
            
            result = self.transform(temp_files)
            
            for path in temp_files.values():
                if os.path.exists(path):
                    os.unlink(path)
            
            return result
    
    transforms = Compose([
        LoadImaged(keys=["fixed", "moving", "fixed_label", "moving_label"]),
        EnsureChannelFirstd(keys=["fixed", "moving", "fixed_label", "moving_label"]),
        ScaleIntensityRanged(
            keys=["fixed", "moving"],
            a_min=CT_MIN_HU, a_max=CT_MAX_HU,
            b_min=0.0, b_max=1.0, clip=True
        ),
        Resized(keys=["fixed", "moving"], spatial_size=config["spatial_size"], mode="trilinear"),
        Resized(keys=["fixed_label", "moving_label"], spatial_size=config["spatial_size"], mode="nearest"),
    ])
    
    train_ds = StageDataset(train_files, transforms)
    val_ds = StageDataset(val_files, transforms)
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=1, num_workers=0)
    
    model = LocalNet(
        spatial_dims=3, in_channels=2, out_channels=3,
        num_channel_initial=config["num_channel_initial"],
        extract_levels=[3], out_activation=None, out_kernel_initializer="zeros"
    ).to(device)
    
    warp = Warp().to(device)
    loss_sim = GlobalMutualInformationLoss()
    loss_reg = BendingEnergyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])
    
    best_dice = 0.0
    
    for epoch in range(config["max_epochs"]):
        model.train()
        epoch_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            fixed = batch["fixed"].to(device)
            moving = batch["moving"].to(device)
            
            ddf = model(torch.cat([moving, fixed], dim=1))
            warped = warp(moving, ddf)
            
            loss = loss_sim(warped, fixed) + config["reg_weight"] * loss_reg(ddf)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        model.eval()
        dice_scores = []
        with torch.no_grad():
            for vb in val_loader:
                vf = vb["fixed"].to(device)
                vm = vb["moving"].to(device)
                vfl = vb["fixed_label"].to(device)
                vml = vb["moving_label"].to(device)
                
                vddf = model(torch.cat([vm, vf], dim=1))
                pred = warp(vml, vddf)
                dice = (2 * (pred * vfl).sum() + 1e-5) / (pred.sum() + vfl.sum() + 1e-5)
                dice_scores.append(dice.item())
        
        avg_dice = np.mean(dice_scores) if dice_scores else 0
        print(f"Epoch {epoch+1}/{config['max_epochs']}: loss={epoch_loss:.4f}, dice={avg_dice:.4f}")
        
        if avg_dice > best_dice:
            best_dice = avg_dice
    
    session = get_active_session()
    local_path = "/tmp/monai_model.pth"
    torch.save(model.state_dict(), local_path)
    session.file.put(local_path, save_stage, overwrite=True, auto_compress=False)
    os.unlink(local_path)
    
    print(f"Model saved to {save_stage}/monai_model.pth")
    return {"best_dice": best_dice, "model_path": f"{save_stage}/monai_model.pth"}

## Launch Training Job

In [12]:
print(f"Submitting training job to {COMPUTE_POOL}...")
print(f"  Cases: {len(train_split)} train, {len(val_split)} val")
print(f"  Epochs: {CONFIG['max_epochs']}")

job = train_model(
    train_files=train_split,
    val_files=val_split,
    config=CONFIG,
    save_stage=RESULTS_STAGE
)

print(f"Job submitted: {job.id}")

Submitting training job to MONAI_GPU_POOL...
  Cases: 16 train, 4 val
  Epochs: 15
Job submitted: MONAI_QUICKSTART_DB.ML.TRAIN_MODEL_1L37PZN1UAOCS


In [13]:
# Monitor job
while job.status not in ["DONE", "FAILED"]:
    print(f"Status: {job.status}")
    time.sleep(30)

if job.status == "DONE":
    result = job.result()
    print(f"Training complete! Best Dice: {result['best_dice']:.4f}")
    print(f"Model path: {result['model_path']}")
else:
    print(f"Training failed: {job.get_logs()}")

Status: PENDING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Training complete! Best Dice: 0.8089
Model path: @MODEL_ARTIFACTS_STG/monai_model.pth


## Load Trained Model and Prepare Sample Input

In [14]:
from monai.networks.nets import LocalNet

# Download model weights
model_dir = tempfile.mkdtemp()
session.file.get(f"{RESULTS_STAGE}/monai_model.pth", model_dir)

# Reconstruct model
model = LocalNet(
    spatial_dims=3, in_channels=2, out_channels=3,
    num_channel_initial=CONFIG["num_channel_initial"],
    extract_levels=[3], out_activation=None, out_kernel_initializer="zeros"
)
model.load_state_dict(torch.load(f"{model_dir}/monai_model.pth", map_location="cpu"))
model.eval()

print("Model loaded")

Model loaded


/var/folders/wh/1t1qbkgx1csgf4q8fxkt5gnr0000gn/T/ipykernel_38546/2735395514.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"{model_di

In [15]:
# Preprocess validation data (already downloaded earlier)
fixed = preprocess_ct(LOCAL_FIXED_PATH, CONFIG["spatial_size"])
moving = preprocess_ct(LOCAL_MOVING_PATH, CONFIG["spatial_size"])

# Create sample input: [batch=1, channels=2, D, H, W]
sample_input = torch.cat([moving, fixed], dim=0).unsqueeze(0)

print(f"Sample input from validation data ({val_case['case_id']}): {sample_input.shape}")

# Verify model works
with torch.no_grad():
    sample_output = model(sample_input)

print(f"Sample output: {sample_output.shape}")

Sample input from validation data (case_017): torch.Size([1, 2, 96, 96, 96])
Sample output: torch.Size([1, 3, 96, 96, 96])


## Log Model to Registry

In [16]:
registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

MODEL_NAME = "MONAI_LUNG_CT_REGISTRATION"
VERSION = "v1"

print(f"Logging model: {MODEL_NAME}/{VERSION}")

mv = registry.log_model(
    model=model,
    model_name=MODEL_NAME,
    version_name=VERSION,
    sample_input_data=sample_input,
    pip_requirements=["monai>=1.3.0", "nibabel", "torch>=2.0.0"],
)

print(f"Model logged: {mv.fully_qualified_model_name}")

python(41492) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Logging model: MONAI_LUNG_CT_REGISTRATION/v1
Logging model: validating model and dependencies...:   0%|          | 0/6 [00:00<?, ?it/s]

/opt/anaconda3/envs/snowml-local/lib/python3.12/site-packages/snowflake/ml/registry/_manager/model_parameter_reconciler.py:155: UserWarning: Models logged specifying `pip_requirements` cannot be executed in a Snowflake Warehouse without specifying `artifact_repository_map`. This model can be run in Snowpark Container Services. See https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/container.
  warnings.warn(


Logging model: creating model manifest...:  33%|███▎      | 2/6 [00:03<00:07,  1.86s/it]  

/opt/anaconda3/envs/snowml-local/lib/python3.12/site-packages/snowflake/ml/model/_packager/model_env/model_env.py:149: UserWarning: Dependencies specified from pip requirements. This may prevent model deploying to Snowflake Warehouse. Use 'artifact_repository_map' to deploy the model to Warehouse.
  self._warn_once(


Model logged successfully.: 100%|██████████| 6/6 [01:57<00:00, 19.61s/it]                          
Model logged: MONAI_QUICKSTART_DB.ML.MONAI_LUNG_CT_REGISTRATION


## Verify Signature

In [17]:
for func in mv.show_functions():
    print(f"Function: {func['name']}")
    sig = func['signature']
    print(f"  Inputs: {[(f.name, f._shape) for f in sig.inputs]}")
    print(f"  Outputs: {[(f.name, f._shape) for f in sig.outputs]}")

Function: FORWARD
  Inputs: [('input_feature_0', (96, 96, 96)), ('input_feature_1', (96, 96, 96))]
  Outputs: [('output_feature_0', (96, 96, 96)), ('output_feature_1', (96, 96, 96)), ('output_feature_2', (96, 96, 96))]


In [18]:
session.close()
print("Done! Proceed to 04_batch_inference.ipynb")

Done! Proceed to 04_batch_inference.ipynb
